# 04 — Testing & scenario comparison (Step 4)
Compare **NN vs RF** across the closest-users scenarios `X ∈ {3,5,10}` (users sampled at random from the
full ACC Arena population), on MSE / MAE / R² and training duration, and identify the best configuration.

In [ ]:
# === Project config — Team 8: Throughput Prediction in a Dense 5G deployment (with Transfer Learning) ===
RANDOM_SEED      = 42
RESAMPLE_SECONDS = 60            # downsampling step in seconds. Lower = more samples, more time (try 30 or 15).
N_USERS          = 1500          # ACC Arena users, sampled at RANDOM from the full ~12k population (seeded).
                                 # Closest-users neighbours are searched WITHIN this sample.
N_USERS_SALT     = 3000          # Salt&Tar users: ALL of them (only ~1/3 are ever active; activity is id-biased)
X_VALUES         = [3, 5, 10]    # number of closest users to experiment with. X=0/1 excluded by design:
                                 # heavy co-location makes a single arbitrary neighbour uninformative.
BEST_X           = 3             # X used for the transfer-learning experiment (pick the best from notebook 04)
OUTLIER_PCT      = 99.0          # drop samples with throughput above this TRAIN percentile (None = keep all).
                                 # EDA (notebook 01): the ~1% extreme samples carry ~2/3 of the total variance.
ACTIVE_ONLY      = True           # regress only on ACTIVE users; idle/off have throughput ~0 by definition
MIN_TRAFFIC_TYPE = 2             # keep traffic_type >= this (2=const_rate, 3=video, 4=gaming, 5=http)

# --- data location ---
DRIVE_FOLDER_ID  = "1s1YCWyhN_Fv5sQraTVs4Rga-ATiKPRgo"   # shared Drive folder containing the zip
ZIP_NAME         = "L5GHDD_Dataset.zip"
DATA_ROOT        = "data/raw"     # dataset folders live here after unzip (local default)
PROCESSED_DIR    = "data/processed"
RESULTS_DIR      = "results"

import os, numpy as np, random
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/figures", exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/models", exist_ok=True)

In [ ]:
# === Colab: mount Drive and make outputs PERSIST across notebooks (no-op locally) ===
def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

if in_colab():
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR   = "/content/drive/MyDrive/5g_throughput_team8"   # persistent project folder on your Drive
    PROCESSED_DIR = f"{PROJECT_DIR}/processed"                     # 02 writes here, 03/04/05 read from here
    RESULTS_DIR   = f"{PROJECT_DIR}/results"                       # models, metrics.csv, figures
    print("Outputs persist in:", PROJECT_DIR)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/figures", exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/models", exist_ok=True)

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
res = pd.read_csv(f"{RESULTS_DIR}/metrics.csv")
res.sort_values(["model","X"])

## Metrics vs X

In [ ]:
metrics = ["MSE","MAE","R2","train_s"] + (["infer_ms"] if "infer_ms" in res.columns else [])
fig, ax = plt.subplots(1, len(metrics), figsize=(4*len(metrics),4))
for i, metric in enumerate(metrics):
    for model in ["NN","RF"]:
        d = res[res.model==model].sort_values("X")
        ax[i].plot(d.X, d[metric], marker="o", label=model)
    ax[i].set_xlabel("X (closest users)"); ax[i].set_title(metric); ax[i].legend(); ax[i].grid(alpha=.3)
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/figures/04_metrics_vs_X.png", dpi=120); plt.show()

## Best configuration and pred-vs-true

In [ ]:
best = res.loc[res.R2.idxmax()]
print("Best:", best[["model","X","MSE","MAE","R2","train_s"]].to_dict())

import numpy as np, joblib
from tensorflow import keras
x = int(best.X)
d = np.load(f"{PROCESSED_DIR}/acc_X{x}.npz"); Xte, yte = d["X_test"], d["y_test"]
if best.model == "NN":
    pred = keras.models.load_model(f"{RESULTS_DIR}/models/nn_X{x}.keras").predict(Xte, verbose=0).ravel()
else:
    pred = joblib.load(f"{RESULTS_DIR}/models/rf_X{x}.pkl").predict(Xte)

plt.figure(figsize=(5,5))
plt.scatter(yte, pred, s=5, alpha=.3)
lim = [0, max(yte.max(), pred.max())]; plt.plot(lim, lim, "r--")
plt.xlabel("True throughput (Mbps)"); plt.ylabel("Predicted"); plt.title(f"{best.model} (X={x})")
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/figures/04_pred_vs_true.png", dpi=120); plt.show()

## How much do the closest-users features contribute? RF feature importance
Aggregate importance of the user's own features vs the neighbour (`nb*`) features for the mid scenario.

In [ ]:
import json as _json
x_imp = 5                                     # scenario to inspect
rf = joblib.load(f"{RESULTS_DIR}/models/rf_X{x_imp}.pkl")
cols = _json.load(open(f"{PROCESSED_DIR}/acc_X{x_imp}_cols.json"))
imp = pd.Series(rf.feature_importances_, index=cols).sort_values(ascending=False)
nb_mask = imp.index.str.startswith("nb")
print(f"aggregate importance:  own features {imp[~nb_mask].sum()*100:.1f}%  |  "
      f"neighbour features {imp[nb_mask].sum()*100:.1f}%  (spread over {nb_mask.sum()} columns)")
plt.figure(figsize=(8,5))
imp.head(15)[::-1].plot.barh(color=["#e76f51" if c.startswith("nb") else "#2a9d8f" for c in imp.head(15)[::-1].index])
plt.title(f"RF feature importance, X={x_imp} (green = own, red = neighbours)")
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/figures/04_feature_importance.png", dpi=120); plt.show()

**Interpretation notes (context from the raw-data analysis).** Users are placed on discrete spots and heavily
co-located (at a given instant, tens of users can share the exact same position), and throughput in this
dataset is strongly **demand-driven**: the user's own `prb` dominates, while neighbour columns share the
remainder. When comparing X = 3/5/10, look at (a) whether the aggregate neighbour importance grows with X,
(b) whether R² actually improves or the extra columns only dilute the RF's per-split feature sampling
(`max_features="sqrt"`), and (c) the training-cost increase. Users are now sampled **at random from the whole
venue**, so the neighbour sample is unbiased with respect to user ids.

## Takeaways
- Compare how the metrics move across `X ∈ {3,5,10}` for both models (NN vs RF).
- Note the **accuracy vs training-cost** trade-off (training duration grows with X / model complexity).
- Set `BEST_X` in the config of notebook **05** to the best-performing X for the transfer-learning study.